In [ ]:
import pandas as pd
from rake_nltk import Rake
from collections import Counter
import nltk
import spacy
from nltk import word_tokenize
from spacy.matcher.matcher import Matcher
import string
import re
import unidecode
import swifter

class Preprocessing:
    def deEmojify(self,inputString):
        returnString = ""

        for character in inputString:
            try:
                character.encode("ascii")
                returnString += character
            except UnicodeEncodeError:
                try:
                    replaced = unidecode(str(character))
                    if replaced != '':
                        returnString += replaced
                except:
                    pass
                    # print("character:", character)
        return returnString


    def handle_emojis(self,tweet):#A made changes
        # Smile -- :), : ), :-), (:, ( :, (-:, :')
        tweet = re.sub(r'(:\s?\)|:-\)|\(\s?:|\(-:|:\'\))', '', tweet)
        # Laugh -- :D, : D, :-D, xD, x-D, XD, X-D
        tweet = re.sub(r'(:\s?D|:-D|x-?D|X-?D)', '', tweet)
        # Love -- <3, :*
        tweet = re.sub(r'(<3|:\*)', '', tweet)
        # Wink -- ;-), ;), ;-D, ;D, (;,  (-;
        tweet = re.sub(r'(;-?\)|;-?D|\(-?;)', '', tweet)
        # Sad -- :-(, : (, :(, ):, )-:
        tweet = re.sub(r'(:\s?\(|:-\(|\)\s?:|\)-:)', '', tweet)
        # Cry -- :,(, :'(, :"(
        tweet = re.sub(r'(:,\(|:\'\(|:"\()', '', tweet)
        return tweet


    def preprocess_word(self,word):
        word = re.sub(r'(\w)\1{3,}', r'\1\1', word)
        return word


    def is_valid_word(self,word):
        return True

    def preprocess_tweet(self,tweet):
        # print tweet
        processed_tweet = []
        # Convert to lower case
        tweet=str(tweet)
        tweet = tweet.lower().strip()

        # Remove unicode emojis
        tweet = self.deEmojify(tweet)
        # Replace ’ character with '
        tweet = re.sub('’', '\'', tweet)

        # replace &amp; and &gt; and &lt;
        tweet = re.sub(r'&amp;', 'and', tweet)
        tweet = re.sub(r'&gt;', '>', tweet)
        tweet = re.sub(r'&lt;', '<', tweet)

        # Replaces URLs with <url>
        #tweet = re.sub(r'((www\.[\S]+)|(https?://[\S]+))', '<url>', tweet) 
        tweet = re.sub(r'((www\.[\S]+)|(https?://[\S]+))', '', tweet) 
        # Replace @handle with the name of the user
        tweet = re.sub(r'@([\S]+)', r'', tweet) # re.sub(r'@([\S]+)', r'\1', tweet)
        # Replaces #hashtag with hashtag
        #tweet = re.sub(r'#(\S+)', r'', tweet) # re.sub(r'#(\S+)', r'\1', tweet) #commented A
        tweet = re.sub(r'#(\S+)', r'', tweet) 
        # Remove RT (retweet) from beginning
        tweet = re.sub(r'^\brt\b', '', tweet)
        # Remove RT inside the tweet
        tweet = re.sub(r' rt ', '', tweet)
        # Replace 2+ dots with space
        tweet = re.sub(r'\.{2,}', ' ', tweet)
        # Strip space, " and ' from tweet
        tweet = tweet.strip(' "\'')
        # Replace emojis with either EMO_POS or EMO_NEG
        tweet = self.handle_emojis(tweet)
        # Replace multiple spaces with a single space
        tweet = re.sub(r'\s+', ' ', tweet)
        tweet = re.sub(r'[",():;\-/“]', '', tweet)

        tweet = re.sub(r' \.', '.', tweet)
        tweet = re.sub(r'’', '\'', tweet)

        words = nltk.word_tokenize(tweet)
        # print ' '.join(words )
        for word in words:
            word = self.preprocess_word(word)
            if self.is_valid_word(word):
                processed_tweet.append(word)
        tweet = ' '.join(processed_tweet)

        tweet = tweet.replace("< ", "<")
        tweet = tweet.replace(" >", ">")
        # Replace nt with not
        tweet = re.sub(r'\bnt\b', 'not', tweet)
        return tweet

def extract_key_phrases(tweets):
    
    nlp = spacy.load("en_core_web_sm")
    
    
    entity_types = {}
    
    for sentence in tweets:
        
        doc = nlp(sentence)
        
        
        for ent in doc.ents:
            entity = ent.text
            entity_type = ent.label_
            
            
            if entity in entity_types:
                if entity_type not in entity_types[entity]:
                    entity_types[entity].append(entity_type)
            else:
                entity_types[entity] = [entity_type]
    
    
    entities_and_types = [(entity, ','.join(types)) for entity, types in entity_types.items()]
    
    return entities_and_types
def main():
    df = pd.read_csv('llm_synthetic_tweets.csv')
    df.rename(columns={'response': 'content'}, inplace=True)

    print(df.head())

    
    print(df.shape)
    
    en_df = df#[df['lang'] == 'en']
    print(en_df.head())
    print(en_df.shape)

    
    
    unique_content_values = en_df['content'].tolist()
    print(len(unique_content_values))
    print(unique_content_values[:4])
    print("length")
    print(len(unique_content_values))
    merged_df = pd.DataFrame(unique_content_values, columns=['content'])
    preprocess=Preprocessing()
    merged_df['processed_tweet_text'] = merged_df['content'].swifter.apply(lambda x: preprocess.preprocess_tweet(x))
    merged_df['processed_tweet_text']=merged_df['processed_tweet_text'].str.strip() 
    merged_df = merged_df[merged_df['processed_tweet_text'] != '']
    unique_content_values = merged_df['processed_tweet_text'].tolist()
    a=extract_key_phrases(unique_content_values)
    df = pd.DataFrame(a, columns=['Entity', 'Types'])
    df.to_csv('spacy_ne.csv', index=False)
if __name__ == "__main__":
    main()
